In [ ]:
# Example using epicardial cells (EPI)
# Repeat with other cardiac cell types

# Import Python packages

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sceptic import run_sceptic_and_evaluate

In [ ]:
# Load subsetted raw counts (see subset_mats_psupertime folder hosted on Zenodo)

adata = sc.read_10x_mtx("subset_mats_psupertime/EPI_subset_mat", var_names = 'gene_ids', cache=True)

# Normalize data

counts = sc.pp.normalize_total(adata, target_sum=None, inplace=False)

# Make array for sceptic analysis

array = counts["X"].toarray()
integer_array = array.astype(np.int32)

# Make data frame for SHAP analysis

mybarcodes = adata.obs.index.values
mygenes = adata.var["gene_symbols"].values
df = pd.DataFrame(array)
df.index = mybarcodes
df.columns = mygenes

In [ ]:
# Load metadata (see subset_meta_psupertime folder hosted on Zenodo)

meta = pd.read_csv('subset_meta_psupertime/EPI_meta.txt', sep='\t')

# Extract timepoints and convert to "real" time in no. of days (referred to as encoded)
time_labels_encoded = meta["Age"].values
time_labels_encoded[time_labels_encoded == 'E14.5'] = 14
time_labels_encoded[time_labels_encoded == 'E16.5'] = 16
time_labels_encoded[time_labels_encoded == 'E18.5'] = 18
time_labels_encoded[time_labels_encoded == 'P0'] = 20
time_labels_encoded[time_labels_encoded == 'P4'] = 24
time_labels_encoded[time_labels_encoded == 'P7'] = 27

integer_time = time_labels_encoded.astype(np.int32)

In [ ]:
# Run sceptic analysis

cm, pred, pseudotime, prob = run_sceptic_and_evaluate(
    data=integer_array,
    labels=integer_time,  # Pass actual hours directly!
    method="xgboost"
)

print("Confusion Matrix:")
print(cm)
print(f"\nPseudotime range: {pseudotime.min():.1f} - {pseudotime.max():.1f} days")

In [ ]:
# Save resulting pseudotime values
np.savetxt('sceptic_pseudotime_EPI.csv', pseudotime, delimiter=',')

In [ ]:
# Note: Can proceed to next step or use file hosted on GitHub (in sceptic_pseudotime folder)
# However there will likely still be some variance in subsequent SHAP results

# pseudotime = pd.read_csv("sceptic_pseudotime/sceptic_pseudotime_EPI.csv", header=None)
# pseudotime = pseudotime.to_numpy()

In [ ]:
# SHAP analysis

import shap
import xgboost as xgb
from sklearn.model_selection import train_test_split

# 1. Train a model to predict pseudotime from gene expression
X_train, X_test, y_train, y_test = train_test_split(df, pseudotime, test_size=0.2)
model = xgb.XGBRegressor().fit(X_train, y_train)

# 2. Use SHAP to explain the model
explainer = shap.Explainer(model)
shap_values = explainer(X_test)

In [ ]:
# Generate barplot of mean SHAP values

fig = shap.plots.bar(shap_values.abs.mean(0), max_display=11, show=False)
plt.savefig("shap_plot_EPI.png", bbox_inches='tight', dpi=300)

In [ ]:
# Generate pseudotime plot comparable to psupertime 
# Note plot_dt was extracted from psupertime results object (see psupertime_results folder hosted on Zenodo)
# Use file hosted on GitHub (in plot_dt folder) to reproduce results 
# The column "psuper" represents pseudotime scores from psupertime
# The column "value" represents z-scored log2 expression for H19
# The column "Pseudotime" represents added pseudotime scores from sceptic 

from plotnine import *

plot_dt= pd.read_csv('plot_dt/plot_dt_EPI.txt', sep='\t')
col_vals = ["#3288BD", "#99D594", "#E6F598", "#FEE08B", "#FC8D59", "#D53E4F"]
fig = (
    ggplot(plot_dt) +
    aes( x=plot_dt["Pseudotime"], y=plot_dt["value"] ) +
    geom_point(aes(colour=plot_dt["Labels"]) ) +
    geom_smooth(method="loess", colour='black') +
    labs(x="Pseudotime (Days)", y="H19 z-scored log2 expression") +
    theme_bw() +
    scale_colour_manual( values=col_vals ) +
    theme(
        axis_title=element_text(size=18),
        axis_text_x=element_text(size=12),
        axis_text_y=element_text(size=12),
        legend_title=element_text(size=16),
        legend_text=element_text(size=14),
        legend_key_size=16
    )
    )
fig
fig.save("sceptic_plot_EPI.png", width=6, height=5, dpi=300)